In [2]:
# Libraries importations :
import numpy as np 
import pandas as pd

import plotly.graph_objs as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from tqdm import tqdm
tqdm.pandas()

import warnings
warnings.filterwarnings("ignore")
import gc

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

import optuna

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

bold = ['\033[1m', '\033[0m']

# Read files
path = '/kaggle/input/playground-series-s3e1/'
train = pd.read_csv(path+'train.csv')
test = pd.read_csv(path+'test.csv')
ss = pd.read_csv(path+'sample_submission.csv')

from sklearn.datasets import fetch_california_housing as fch
sklearn_df = pd.DataFrame(fch()['data'], columns=fch()['feature_names'])
sklearn_df['MedHouseVal'] = fch()['target']

# Show all columns
pd.set_option('display.max_columns', None)


---

# | Preprocessing

---

## || Generated Data

In [3]:
sklearn_df['is_generated'] = 0
train['is_generated'] = 1
test['is_generated'] = 1

train.drop('id', axis=1, inplace=True)
train = pd.concat([train, sklearn_df], 
                  ignore_index=True)
test.drop('id', axis=1, inplace=True)

train.loc[33228,['Latitude','Longitude']] = [32.74, -117]
train.loc[34363,['Latitude','Longitude']] = [32.71, -117]
train.loc[20991,['Latitude','Longitude']] = [34.2, -119]

df = pd.concat([train, test], axis=0, ignore_index=True)

## || Encoding trick

In [4]:
emb_size = 20
precision = 1e6 

latlon = np.expand_dims(df[['Latitude', 'Longitude']].values, axis=-1) 

m = np.exp(np.log(precision) / emb_size) 
angle_freq = m ** np.arange(emb_size) 
angle_freq = angle_freq.reshape(1, 1, emb_size) 

latlon = latlon * angle_freq  # Frequence de corrélation angulatoires triangulées
latlon[..., 0::2] = np.cos(latlon[..., 0::2]) 
latlon[..., 1::2] = np.sin(latlon[..., 1::2]) 
latlon = latlon.reshape(-1, 2 * emb_size) 

In [5]:
df['exp_latlon1'] = [lat[0] for lat in latlon]
df['exp_latlon2'] = [lat[1] for lat in latlon]

In [6]:
from sklearn.decomposition import PCA

coordinates = df[['Latitude', 'Longitude']].values
pca = PCA().fit(coordinates)
df['pca_lat'] = pca.transform(coordinates)[:,0]
df['pca_lon'] = pca.transform(coordinates)[:,1]

In [7]:
%%time

from umap import UMAP
umap = UMAP(n_components=2, n_neighbors=50, random_state=228).fit(coordinates)
df['umap_lat'] = umap.transform(coordinates)[:,0]
df['umap_lon'] = umap.transform(coordinates)[:,1]

CPU times: user 13min 35s, sys: 2min 42s, total: 16min 18s
Wall time: 8min 13s


---

# Pycaret anomaly /ToDo : Try to plot UMAP

In [51]:
pip install pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.2/320.2 kB 548.9 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.7/147.7 kB 3.0 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.9/25.9 MB 13.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 13.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 20.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 636.6/636.6 kB 37.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 16.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.1/167.1 kB 15.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 17.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 15.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.6 MB/s eta 0:00:

In [52]:
from pycaret.anomaly import *

ModuleNotFoundError: No module named 'pycaret'

In [ ]:
_ = pycaret.anomaly.setup(
    data=space_df[tr_idx],
    silent=True)

In [ ]:
knn = pycaret.anomaly.create_model('knn')

In [53]:
knn_df = pycaret.anomaly.assign_model(knn)
abnormal_data = knn_df[knn_df.Anomaly == 1].sort_values(by='Anomaly_Score', ascending=False)
print("the size of anomaly = ",len(abnormal_data))
abnormal_data.head().style.set_properties(**{'background-color': 'black',
                           'color': 'white',
                           'border-color': 'white'})

NameError: name 'pycaret' is not defined

In [49]:
plt.style.use("dark_background")
plot_model(knn,plot='umap')

NameError: name 'plot_model' is not defined

---

## || Cartesian coordinates rotation

---

In [8]:
def crt_crds(df):
    
    df['rot_15_x'] = (np.cos(np.radians(15)) * df['Longitude']) + \
                      (np.sin(np.radians(15)) * df['Latitude'])
    
    df['rot_15_y'] = (np.cos(np.radians(15)) * df['Latitude']) + \
                      (np.sin(np.radians(15)) * df['Longitude'])
    
    df['rot_30_x'] = (np.cos(np.radians(30)) * df['Longitude']) + \
                      (np.sin(np.radians(30)) * df['Latitude'])
    
    df['rot_30_y'] = (np.cos(np.radians(30)) * df['Latitude']) + \
                      (np.sin(np.radians(30)) * df['Longitude'])
    
    df['rot_45_x'] = (np.cos(np.radians(45)) * df['Longitude']) + \
                      (np.sin(np.radians(45)) * df['Latitude'])
    return df

# train_df = crt_crds(train_df)
# test_df = crt_crds(test_df)
df = crt_crds(df)

In [18]:
df['rot_45_x'] = (np.cos(np.radians(45)) * df['Longitude']) + \
                  (np.sin(np.radians(45)) * df['Latitude'])
    
df['rot_45_y'] = (np.cos(np.radians(45)) * df['Latitude']) + \
                  (np.sin(np.radians(45)) * df['Longitude'])
    
df['rot_135_x'] = (np.cos(np.radians(135)) * df['Longitude']) + \
                  (np.sin(np.radians(135)) * df['Latitude'])
    
df['rot_135_y'] = (np.cos(np.radians(135)) * df['Latitude']) + \
                  (np.sin(np.radians(135)) * df['Longitude'])
    
df['rot_270_x'] = (np.cos(np.radians(270)) * df['Longitude']) + \
                  (np.sin(np.radians(270)) * df['Latitude'])

df['rot_270_y'] = (np.cos(np.radians(270)) * df['Longitude']) + \
                  (np.sin(np.radians(270)) * df['Latitude'])

## || Location of coordinates

In [9]:
pip install reverse_geocoder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 4.6 MB/s eta 0:00:0000:0100:010m
  Preparing metadata (setup.py) ... done
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268088 sha256=095f2f99ed139fc20b0597946f4d38c300e493ac9959675705b38aba3da53464
  Stored in directory: /root/.cache/pip/wheels/34/6e/70/5423639428a2cac8ea7eb467214a4254b549b381f306a9c790
Successfully built reverse_geocoder
Note: you may need to restart the kernel to use updated packages.


In [10]:
import reverse_geocoder as rg

coordinates = list(zip(df['Latitude'], df['Longitude']))
results = rg.search(coordinates)
df['place'] = [x['admin2'] for x in results]

places = ['Los Angeles County', 'Orange County', 'Kern County',
          'Alameda County', 'San Francisco County', 'Ventura County',
          'Santa Clara County', 'Fresno County', 'Santa Barbara County',
          'Contra Costa County', 'Yolo County', 'Monterey County',
          'Riverside County', 'Napa County']

def replace(x):
    if x in places:
        return x
    else:
        return 'Other'
    
df['place'] = df['place'].apply(lambda x: replace(x))
le = LabelEncoder()
df['place'] = le.fit_transform(df['place'])

Loading formatted geocoded file...


## || Distance to cities, coast points & parcs

In [12]:
from haversine import haversine

Sac = (38.576931, -121.494949)
SF = (37.780080, -122.420160)
SJ = (37.334789, -121.888138)
LA = (34.052235, -118.243683)
SD = (32.715759, -117.163818)

Yosemite = (37.866747, -119.538076)
Sequoia = (36.489057, -118.565255)
Kings_Canyon = (36.887855, -118.555146)
Death_Valley = (36.464288, -116.869045)
Joshua_Tree = (34.133831, -116.314867)

# Biggers Towns :
df['dist_Sac'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), Sac, unit='ft'), axis=1)
df['dist_SF'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), SF, unit='ft'), axis=1)
df['dist_SJ'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), SJ, unit='ft'), axis=1)
df['dist_LA'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), LA, unit='ft'), axis=1)
df['dist_SD'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), SD, unit='ft'), axis=1)

# Biggers Parks :
df['dist_Yosemite'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), Yosemite, unit='ft'), axis=1)
df['dist_Sequoia'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), Sequoia, unit='ft'), axis=1)
df['dist_Kings_Canyon'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), Kings_Canyon, unit='ft'), axis=1)
df['dist_Death_Valley'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), Death_Valley, unit='ft'), axis=1)
df['dist_Joshua_Tree'] = df.apply(lambda x: haversine((x['Latitude'], x['Longitude']), Joshua_Tree, unit='ft'), axis=1)

df['dist_nearest_city'] = df[['dist_Sac', 'dist_SF', 'dist_SJ', 'dist_LA', 'dist_SD']].min(axis=1)
df['dist_nearest_parc'] = df[['dist_Yosemite', 'dist_Sequoia', 'dist_Kings_Canyon','dist_Death_Valley', 'dist_Joshua_Tree']].min(axis=1)

In [18]:
import folium
from folium import plugins
from folium.plugins import HeatMap

In [19]:
heat_map = folium.Map(train[['Latitude', 'Longitude']].mean(axis=0),
                    zoom_start = 6) 

train['Latitude'] = train['Latitude'].astype(float)
train['Longitude'] = train['Longitude'].astype(float)

lat_long_list = [[row['Latitude'],row['Longitude']] for index, row in train.iterrows()]
HeatMap(lat_long_list, radius=10).add_to(heat_map)
heat_map

In [20]:
from shapely.geometry import LineString, Point

coast_points = LineString([(32.6644, -117.1613), (33.2064, -117.3831),
                           (33.7772, -118.2024), (34.4634, -120.0144),
                           (35.4273, -120.8819), (35.9284, -121.4892),
                           (36.9827, -122.0289), (37.6114, -122.4916),
                           (38.3556, -123.0603), (39.7926, -123.8217),
                           (40.7997, -124.1881), (41.7558, -124.1976)])

df['dist_to_coast'] = df.apply(lambda x: Point(x['Latitude'], x['Longitude']).distance(coast_points), axis=1)

df['dist_to_'] = df.apply(lambda x: Point(x['Latitude'], x['Longitude']).distance(coast_points), axis=1)



# 📌 Improvement possible with 2 differents variable creation : string Lines X 
# 📌                                                            string Lines +


## || Final preprocessing

In [21]:
train = df.iloc[:-len(test),:]
test = df.iloc[-len(test):,:].drop('MedHouseVal', axis=1).reset_index(drop=True)

X = train.drop('MedHouseVal', axis=1)
y = train['MedHouseVal']

seed = 228
FOLDS = 10

In [22]:
def rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())

def f_importance_plot(f_imp):
    fig = plt.figure(figsize = (15, 0.35*len(f_imp)))
    plt.title('Feature importances', size=25, y=1.05, 
              fontname='Calibri', fontweight='bold', color='#444444')
    a = sns.barplot(data=f_imp, x='avg_imp', y='feature', 
                    palette='Blues_d', linestyle="-", 
                    linewidth=1, edgecolor="black")
    plt.xlabel('')
    plt.xticks([])
    plt.ylabel('')
    plt.yticks(size=11, color='#444444')
    
    for j in ['right', 'top', 'bottom']:
        a.spines[j].set_visible(False)
    for j in ['left']:
        a.spines[j].set_linewidth(0.5)
    plt.show()

In [23]:
train

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,is_generated,exp_latlon1,exp_latlon2,pca_lat,pca_lon,umap_lat,umap_lon,rot_15_x,rot_15_y,rot_30_x,rot_30_y,rot_45_x,place,dist_Sac,dist_SF,dist_SJ,dist_LA,dist_SD,dist_Yosemite,dist_Sequoia,dist_Kings_Canyon,dist_Death_Valley,dist_Joshua_Tree,dist_nearest_city,dist_nearest_parc,dist_to_coast,dist_to_
0,2.3859,15.0,3.827160,1.112100,1280.0,2.486989,34.60,-120.12,0.980,1,-0.999098,-0.078880,-0.342723,-1.086666,4.196439,11.288356,-107.071871,2.331690,-86.726972,-30.095521,-60.471772,11,1.505644e+06,1.343215e+06,1.125962e+06,5.995613e+05,1.130527e+06,1.203984e+06,8.293757e+05,9.545868e+05,1.180583e+06,1.158322e+06,599561.346601,829375.663564,0.012888,0.012888
1,3.7188,17.0,6.013373,1.054217,1504.0,3.813084,38.69,-121.22,0.946,1,0.547947,0.974193,3.391118,0.912466,17.744240,-1.431677,-107.075820,5.997626,-85.634599,-27.103477,-58.357523,8,8.854832e+04,4.779710e+05,5.303822e+05,1.904151e+06,2.488078e+06,5.676190e+05,1.110590e+06,1.011098e+06,1.496985e+06,2.198480e+06,88548.323144,567618.987857,1.665273,1.665273
2,4.7750,27.0,6.535604,1.103175,1061.0,2.464602,34.71,-120.45,1.576,1,-0.988397,0.140055,-0.036583,-1.251827,8.670680,-1.934410,-107.362157,2.352531,-86.957760,-30.165258,-60.627335,11,1.443453e+06,1.261031e+06,1.047325e+06,7.062707e+05,1.234280e+06,1.182411e+06,8.565765e+05,9.723635e+05,1.240124e+06,1.261930e+06,706270.711605,856576.480476,0.158815,0.158815
3,2.4138,16.0,3.350203,0.965432,1255.0,2.089286,32.66,-117.09,1.336,1,0.320941,0.723030,-3.831383,-0.207032,-2.469708,-14.047390,-104.647225,1.242015,-85.072915,-30.260610,-59.701026,8,2.522266e+06,2.451054e+06,2.227410e+06,6.176880e+05,3.045445e+04,2.034423e+06,1.465434e+06,1.603581e+06,1.389440e+06,5.872150e+05,30454.454611,587214.987893,0.071436,0.071436
4,3.7500,52.0,4.284404,1.069246,1793.0,1.604790,37.80,-122.41,4.500,1,0.994915,0.022690,3.557422,-0.564200,33.871826,13.683404,-108.455620,4.829957,-87.110170,-28.469240,-59.828305,10,3.862307e+05,7.835185e+03,2.271009e+05,1.839177e+06,2.424468e+06,8.278093e+05,1.215906e+06,1.166387e+06,1.683307e+06,2.241369e+06,7835.184568,827809.308562,0.179351,0.179351
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57772,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781,0,-0.208550,-0.231034,3.877808,1.548180,8.378905,7.506400,-106.745782,6.794353,-85.127016,-26.354317,-57.706984,8,3.488665e+05,7.268149e+05,8.151728e+05,2.147412e+06,2.724824e+06,7.360091e+05,1.310428e+06,1.192609e+06,1.637891e+06,2.396890e+06,348866.508021,736009.143611,2.248242,2.248242
57773,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771,0,-0.218319,-0.250399,3.967270,1.467578,8.889896,7.032233,-106.859105,6.772954,-85.225939,-26.405657,-57.784766,8,3.427467e+05,7.127612e+05,8.097853e+05,2.164527e+06,2.743236e+06,7.598418e+05,1.332803e+06,1.216686e+06,1.665952e+06,2.420292e+06,342746.711259,759841.797908,2.158966,2.158966
57774,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923,0,-0.159409,-0.132982,3.930392,1.419204,8.859087,7.053589,-106.884294,6.712411,-85.264599,-26.462618,-57.834264,8,3.208235e+05,6.923452e+05,7.878722e+05,2.145811e+06,2.725036e+06,7.448688e+05,1.316733e+06,1.201712e+06,1.654102e+06,2.404744e+06,320823.532421,744868.778161,2.114589,2.114589
57775,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847,0,-0.159409,-0.132982,3.998871,1.346329,8.975849,6.858022,-106.980886,6.686529,-85.351202,-26.512618,-57.904974,8,3.151381e+05,6.787121e+05,7.814318e+05,2.157805e+06,2.738064e+06,7.635060e+05,1.333606e+06,1.220148e+06,1.675955e+06,2.422230e+06,315138.110115,763505.997825,2.035133,2.035133


---

# Best feature selection with ROC_AUC adapted :

---

# /ToDo : Implémenter MMR pour la feature selection.

# ...

---

# Model 1 : CatBoost

---

In [26]:
import catboost
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

n_folds = 10

MAX_ITER = 22500
PATIENCE = 1500
DISPLAY_FREQ = 100

eval_predsCB = []
predsCB = []

k_fold = KFold(n_splits=n_folds, random_state=51, shuffle = True)

MODEL_PARAMS = {
                'random_seed': 51,    
                'iterations': MAX_ITER,
                'early_stopping_rounds': PATIENCE,
                'use_best_model': True,
                'eval_metric': 'RMSE',
                'verbose': 4500,
               }

In [27]:
X = train.drop(columns=['MedHouseVal'])
y = train.MedHouseVal
X_test = test

In [28]:
for train_index, test_index in k_fold.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[test_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[test_index]
    
    model = catboost.CatBoostRegressor(**MODEL_PARAMS)
    
    model.fit(X=X_train, y=y_train,
          eval_set=[(X_valid, y_valid)],
          early_stopping_rounds = PATIENCE,
         )
    predsCB.append(model.predict(X_test))
#     eval_predsCB.append(model.predict(X))
#     print("RMSE valid = {}".format(mean_squared_error(y_valid, model.predict(X_valid))))
#     print("RMSE full = {}".format(mean_squared_error(y, model.predict(X))))

Learning rate set to 0.014173
0:	learn: 1.1482419	test: 1.1314880	best: 1.1314880 (0)	total: 81.2ms	remaining: 30m 26s
4500:	learn: 0.4610418	test: 0.5019047	best: 0.5019047 (4500)	total: 58.7s	remaining: 3m 54s
9000:	learn: 0.4217495	test: 0.4972462	best: 0.4972332 (8865)	total: 1m 57s	remaining: 2m 56s
Stopped by overfitting detector  (1500 iterations wait)

bestTest = 0.4968257761
bestIteration = 10245

Shrink model to first 10246 iterations.
Learning rate set to 0.014173
0:	learn: 1.1454066	test: 1.1572453	best: 1.1572453 (0)	total: 14.7ms	remaining: 5m 30s
4500:	learn: 0.4592622	test: 0.5247496	best: 0.5247469 (4499)	total: 59s	remaining: 3m 55s
9000:	learn: 0.4206214	test: 0.5214844	best: 0.5214841 (8999)	total: 1m 57s	remaining: 2m 56s
13500:	learn: 0.3920471	test: 0.5207711	best: 0.5207675 (13492)	total: 2m 58s	remaining: 1m 58s
Stopped by overfitting detector  (1500 iterations wait)

bestTest = 0.5207103183
bestIteration = 14742

Shrink model to first 14743 iterations.
Learnin

---

# Model 2 : XGBoost

---

In [39]:
# Extraction de la data pour traitement du tuning XGB au GPU dans un autre Notebook : 
train.to_csv("train_full.csv")

In [38]:
from xgboost import XGBRegressor

n_folds = 9
k_fold = KFold(n_splits=n_folds, random_state=51, shuffle=True)

eval_predsXB = []
predsXB = []

PATIENCE = 200

MODEL_PARAMS = {       'n_estimators': 15000, #1000, 5000
                       #'learning_rate': 0.02,
                       'max_depth': 4,
                       'colsample_bytree': 0.66, # 0.95
                       'subsample': 1,
                       'reg_lambda': 20,
                       'early_stopping_rounds': PATIENCE,
#                      'tree_method': 'gpu_hist',
                       'seed': 1
}

for train_index, test_index in k_fold.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[test_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[test_index]
    
    model = XGBRegressor(**MODEL_PARAMS)
    
    model.fit(X=X_train, y=y_train,
          eval_set=[(X_valid, y_valid)],
#           early_stopping_rounds = PATIENCE,
          verbose = 200
         )
    predsXB.append(model.predict(X_test))
#     eval_predsXB.append(model.predict(X))

[0]	validation_0-rmse:1.48390
[200]	validation_0-rmse:0.51571
[400]	validation_0-rmse:0.51230
[600]	validation_0-rmse:0.51127
[744]	validation_0-rmse:0.51168
[0]	validation_0-rmse:1.51237
[200]	validation_0-rmse:0.53083
[400]	validation_0-rmse:0.52763
[600]	validation_0-rmse:0.52762
[718]	validation_0-rmse:0.52805
[0]	validation_0-rmse:1.53246
[200]	validation_0-rmse:0.52286
[400]	validation_0-rmse:0.51966
[600]	validation_0-rmse:0.52018
[630]	validation_0-rmse:0.52016
[0]	validation_0-rmse:1.48912
[200]	validation_0-rmse:0.51214
[400]	validation_0-rmse:0.51170
[600]	validation_0-rmse:0.51156
[650]	validation_0-rmse:0.51153
[0]	validation_0-rmse:1.52539
[200]	validation_0-rmse:0.52285
[400]	validation_0-rmse:0.52136
[511]	validation_0-rmse:0.52138
[0]	validation_0-rmse:1.51068
[200]	validation_0-rmse:0.53498
[400]	validation_0-rmse:0.53280
[600]	validation_0-rmse:0.53176
[800]	validation_0-rmse:0.53123
[1000]	validation_0-rmse:0.53275
[1025]	validation_0-rmse:0.53266
[0]	validation_0-r

In [61]:
pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.7/289.7 kB 527.9 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.6/226.6 kB 3.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.0/286.0 kB 4.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 15.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.2/802.2 kB 44.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 MB 10.0 MB/s 

In [62]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [68]:
train.dtypes

MedInc               float64
HouseAge             float64
AveRooms             float64
AveBedrms            float64
Population           float64
AveOccup             float64
Latitude             float64
Longitude            float64
MedHouseVal          float64
is_generated           int64
exp_latlon1          float64
exp_latlon2          float64
pca_lat              float64
pca_lon              float64
rot_15_x             float64
rot_15_y             float64
rot_30_x             float64
rot_30_y             float64
rot_45_x             float64
rot_45_y             float64
place                  int64
dist_Sac             float64
dist_SF              float64
dist_SJ              float64
dist_LA              float64
dist_SD              float64
dist_nearest_city    float64
dist_to_coast        float64
dtype: object

In [69]:
train["place"] = train["place"].astype("category")

In [81]:
# --- Launching a TabularPredictor ---

label = 'MedHouseVal'
eval_metric = 'root_mean_squared_error'
preset = 'best_quality'

#ignored_columns  = ["Population","AveBedrms","Latitude","pca_lat","Longitude","rot_15_y","dist_Sac","dist_SJ","rot_30_y","is_generated","rot_30_x","dist_SD","exp_latlon1","dist_SF","rot_15_x"]

predictor = TabularPredictor(label = label,
                             eval_metric = eval_metric,
                             #learner_kwargs = { 'ignored_columns' : ignored_columns }, # For faster feature_importance permutation
                             ).fit(train,
                                   presets = preset,
                                   #hyperparameters = 'light', # Fast research / simplicity of search settings
                                   time_limit = 1 * 3600,
                                   verbosity = 3)

results = predictor.fit_summary()

No path specified. Models will be saved in: "AutogluonModels/ag-20230108_123823/"
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'auto_stack': True, 'verbosity': 3}
Full kwargs:
{'_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': None,
 'auto_stack': True,
 'calibrate': 'auto',
 'excluded_model_types': None,
 'feature_generator': 'auto',
 'feature_prune_kwargs': None,
 'holdout_frac': None,
 'hyperparameter_tune_kwargs': None,
 'keep_only_best': False,
 'name_suffix': None,
 'num_bag_folds': None,
 'num_bag_sets': None,
 'num_stack_levels': None,
 'pseudo_data': None,
 'refit_full': False,
 'save_space': False,
 'set_best_to_refit_full': False,
 'unlabeled_data': None,
 'use_bag_holdout': False,
 'verbosity': 3}
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=20
Saving AutogluonModels/ag-20230108_123823/learner.pkl
Saving 

*** Summary of fit() ***
Estimated performance of each model:
                     model  score_val  pred_time_val     fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      WeightedEnsemble_L3  -0.502684     121.084608  3171.824049                0.001866           0.956919            3       True         14
1        LightGBMXT_BAG_L2  -0.503849     113.663397  2180.199908                3.566422          82.959781            2       True         10
2          LightGBM_BAG_L2  -0.503886     112.441883  2166.605007                2.344908          69.364881            2       True         11
3      WeightedEnsemble_L2  -0.504632      99.402259  1529.287790                0.001977           1.346576            2       True          9
4   RandomForestMSE_BAG_L2  -0.510166     115.171412  3018.542467                5.074437         921.302341            2       True         12
5        LightGBMXT_BAG_L1  -0.511114      64.385175   464.057156         

In [82]:
y_pred = predictor.predict(test)

Loading: AutogluonModels/ag-20230108_123823/models/CatBoost_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/ExtraTreesMSE_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/KNeighborsDist_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/KNeighborsUnif_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/LightGBMXT_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/LightGBM_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/NeuralNetFastAI_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/RandomForestMSE_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/LightGBMXT_BAG_L2/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/LightGBM_BAG_L2/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/RandomForestMSE_BAG_L2/model.pkl
Loading: AutogluonModels/ag-20230108_123823/models/WeightedEnsemble_L3/model.pkl


In [83]:
y_pred

0        0.684733
1        1.075492
2        3.948377
3        3.209853
4        2.460941
           ...   
24754    2.716174
24755    1.794602
24756    1.219339
24757    3.556829
24758    3.420027
Name: MedHouseVal, Length: 24759, dtype: float32

In [86]:
ss["MedHouseVal"] = y_pred

In [88]:
ss = ss.set_index("id")
ss

,MedHouseVal
id,
37137,0.684733
37138,1.075492
37139,3.948377
37140,3.209853
37141,2.460941
...,...
61891,2.716174
61892,1.794602
61893,1.219339


In [89]:
ss.to_csv("sub.csv")

In [ ]:
predictor.feature_importance(train)

In [91]:
# --- Launching a TabularPredictor ---

label = 'MedHouseVal'
eval_metric = 'root_mean_squared_error'
#preset = 'best_quality'

ignored_columns  = ["Population"] # "AveBedrms","Latitude","pca_lat","Longitude","rot_15_y","dist_Sac","dist_SJ","rot_30_y","is_generated","rot_30_x","dist_SD","exp_latlon1","dist_SF","rot_15_x"

predictor = TabularPredictor(label = label,
                             eval_metric = eval_metric,
                             learner_kwargs = { 'ignored_columns' : ignored_columns }, # For faster feature_importance permutation
                             ).fit(train,
                                   presets = preset,
                                   hyperparameters = 'light', # Fast research / simplicity of search settings
                                   time_limit = 0.21 * 3600,
                                   verbosity = 3)

results = predictor.fit_summary()

No path specified. Models will be saved in: "AutogluonModels/ag-20230108_140252/"
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'auto_stack': True, 'verbosity': 3}
Full kwargs:
{'_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': None,
 'auto_stack': True,
 'calibrate': 'auto',
 'excluded_model_types': None,
 'feature_generator': 'auto',
 'feature_prune_kwargs': None,
 'holdout_frac': None,
 'hyperparameter_tune_kwargs': None,
 'keep_only_best': False,
 'name_suffix': None,
 'num_bag_folds': None,
 'num_bag_sets': None,
 'num_stack_levels': None,
 'pseudo_data': None,
 'refit_full': False,
 'save_space': False,
 'set_best_to_refit_full': False,
 'unlabeled_data': None,
 'use_bag_holdout': False,
 'verbosity': 3}
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=20
Saving AutogluonModels/ag-20230108_140252/learner.pkl
Saving 

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val  pred_time_val    fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2  -0.510254      77.958217  498.308474                0.001755           0.691870            2       True          3
1  WeightedEnsemble_L3  -0.511561      86.398510  630.768070                0.001887           0.500617            3       True          6
2    LightGBMXT_BAG_L1  -0.511838      69.981508  407.071790               69.981508         407.071790            1       True          1
3      LightGBM_BAG_L2  -0.512224      81.248441  556.239863                3.291978          58.623260            2       True          5
4    LightGBMXT_BAG_L2  -0.512948      83.104645  571.644194                5.148182          74.027591            2       True          4
5      LightGBM_BAG_L1  -0.518455       7.974954   90.544814                7.974954          90.544814 

In [98]:
y_pred = predictor.predict(test)

Loading: AutogluonModels/ag-20230108_140252/models/LightGBMXT_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_140252/models/LightGBM_BAG_L1/model.pkl
Loading: AutogluonModels/ag-20230108_140252/models/WeightedEnsemble_L2/model.pkl


In [109]:
y_pred = pd.DataFrame(y_pred)

In [115]:
y_pred["MedHouseVal"]

0        0.657503
1        1.043892
2        4.101843
3        3.260886
4        2.508345
           ...   
24754    2.646386
24755    1.876054
24756    1.212681
24757    3.671331
24758    3.598445
Name: MedHouseVal, Length: 24759, dtype: float32

In [117]:
ss = ss.drop(columns=["MedHouseVal"])

In [118]:
ss

""
id
37137
37138
37139
37140
37141
...
61891
61892
61893


In [122]:
y_pred.index = y_pred.index+37137

In [125]:
y_pred.index = y_pred.index.rename("id")

In [128]:
y_pred.to_csv("sub_3.csv")

# Test de LightAutoML :

In [129]:
%%capture
!pip install -U lightautoml

In [130]:
# LightAutoML presets, task and report generation
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task
from lightautoml.report.report_deco import ReportDeco

pd.set_option('display.max_columns', None)

In [131]:
N_THREADS = 4
RANDOM_STATE = 51
# TEST_SIZE = 0.2
TIMEOUT = 1 * 60
TARGET_NAME = 'MedHouseVal'
N_FOLDS = 10

In [132]:
task = Task(name = 'reg',
#             loss = 'mse',
#             metric = 'mse'
           )

In [137]:
roles = {'target': TARGET_NAME,
#          'drop': ['id']
         }

In [138]:
automl = TabularAutoML(task = task,
                       
                       timeout = TIMEOUT,
                       cpu_limit = N_THREADS,
                       reader_params = {'n_jobs': N_THREADS, 'random_state': RANDOM_STATE},
                       general_params = {'use_algos': [['linear_l2', 'lgb', 'cb', 'lgb_tuned','cb_tuned']]}
                      )

In [139]:
oof_pred = automl.fit_predict(train, roles=roles, verbose=5)
print(f'oof_pred:\n{oof_pred}\nShape = {oof_pred.shape}')

[14:38:12] Stdout logging level is DEBUG.
[14:38:12] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer
[14:38:12] Task: reg

[14:38:12] Start automl preset with listed constraints:
[14:38:12] - time: 60.00 seconds
[14:38:12] - CPU: 4 cores
[14:38:12] - memory: 16 GB

[14:38:12] Train data shape: (57777, 28)

[14:38:47] Feats was rejected during automatic roles guess: []
[14:38:47] Layer 1 train process start. Time left 25.14 secs
[14:38:52] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[14:38:52] Training params: {'tol': 1e-06, 'max_iter': 100, 'cs': [1e-05, 5e-05, 0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000], 'early_stopping': 2, 'categorical_idx': [51], 'embed_sizes': array([16], dtype=int32), 'data_size': 52}
[14:38:52] ===== Start working with fold 0 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[14:38:53] Linear model: C = 1e-05 score = -0.42448004392285715
[14:38:53] Linear m

In [140]:
oof_pred_train = oof_pred.data[:len(train), 0]

In [142]:
%%time

test_pred = automl.predict(test)
print(f'Prediction for test data:\n{test_pred}\nShape = {test_pred.shape}')

Prediction for test data:
array([[0.72485113],
       [0.9538561 ],
       [3.9692798 ],
       ...,
       [1.1748868 ],
       [3.855534  ],
       [3.3693113 ]], dtype=float32)
Shape = (24759, 1)
CPU times: user 5.36 s, sys: 45 ms, total: 5.4 s
Wall time: 5.69 s


In [143]:
ss[TARGET_NAME] = test_pred.data[:, 0]

In [144]:
ss

,MedHouseVal
id,
37137,0.724851
37138,0.953856
37139,3.969280
37140,3.114240
37141,2.463113
...,...
61891,2.709436
61892,1.977542
61893,1.174887


In [147]:
ss.to_csv('lightautoml.csv')

---

In [149]:
X = train.copy()
y = train.pop('MedHouseVal')

from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 51)

In [150]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import accuracy_score
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

def predict_and_acc(model):
        model = model()
        model.fit(X_train, y_train)
        predict = model.predict(X_test)
        cvs = cross_val_score(model, X, y, cv = 4)
        print(f'mean_squared_error {str(model)} is {float(mean_squared_error(y_test, predict))}')
        print(f'The cross validation of {str(model)} is:{cvs} with mean of {cvs.mean()}')

In [151]:
predict_and_acc(RandomForestRegressor)

KeyboardInterrupt: 

In [ ]:
predict_and_acc(LGBMRegressor)

In [ ]:
predict_and_acc(CatBoostRegressor)

In [163]:
#object_cols = [col for col in train.columns if train[col].dtype == 'object' or train[col].dtype == 'category']
#train[object_cols] = train[object_cols].astype('category')

In [164]:
train["place"]=train["place"].dty(int)

AttributeError: 'Series' object has no attribute 'as_type'

In [156]:
model = CatBoostRegressor(iterations = 150000)
model.fit(X_train, y_train)
predict = model.predict(X_test)
cvs = cross_val_score(model, X, y, cv = 10)
print(f'mean_squared_error {str(model)} is {float(mean_squared_error(y_test, predict))}')
print(f'The cross validation of {str(model)} is:{cvs} with mean of {cvs.mean()}')

CatBoostError: features data: pandas.DataFrame column 'place' has dtype 'category' but is not in  cat_features list

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import accuracy_score
from catboost import CatBoostRegressor

def predict_and_acc(model, verbose=None):
    if verbose == None:
        model = model()
        model.fit(X_train, y_train)
        predict = model.predict(X_test)
        cvs = cross_val_score(model, X, y, cv=4)
        print(f'The accuracy score of {str(model)} is {float(_score(y_test, predict))}')
        print(f'The cross validation of {str(model)} is:{cvs} with mean of {cvs.mean()}')
    else:
        model = model(verbose=verbose)
        model.fit(X_train, y_train)
        predict = model.predict(X_test)
        cvs = cross_val_score(model, X, y, cv=4)
        print(f'The accuracy score of {str(model)} is {float(accuracy_score(y_test, predict))}')
        print(f'The cross validation of {str(model)} is:{cvs} with mean of {cvs.mean()}')

In [ ]:
predict_and_acc(CatBoostRegressor(iterations = 150000), verbose=15000)

In [ ]:
time_limit = 0.1 * 3600  # for quick demonstration only, you should set this to longest time you are willing to wait (in seconds)
metric = 'roc_auc'  # specify your evaluation metric here
predictor = TabularPredictor(label, eval_metric=metric).fit(train_data, time_limit=time_limit, presets='best_quality', )
predictor.leaderboard(test_data, silent=True)

In [ ]:
predictor = TabularPredictor(label=label, eval_metric=metric).fit(train_data,
    num_bag_folds=5, num_bag_sets=1, num_stack_levels=1,
    hyperparameters = {'NN_TORCH': {'num_epochs': 2}, 'GBM': {'num_boost_round': 20}},  # last  argument is just for quick demo here, omit it in real applications
)

In [ ]:
test_acc = predictor.evaluate(test_data)

KeyError: "['id'] not found in axis"

In [ ]:
# Après modélisation réaliser un ensemble de poids exponentiel pour calculer le poids combinatoire de chaque modèle
# Lors du split, approche temporalepossible séparation du jeu de donnée et des modèles prédictifs en subdivisions découpées par rapport au 
# Créer variable categ N, NE, NO, O, E, S, SE, SO?